# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoyuQre/flyrank_assgn_1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
The unit of analysis is **one content page**, identified by a unique `content_id`. Each row summarizes the historical search-performance, engagement, and content characteristics of a single webpage. Most performance metrics are aggregated over the **previous 90 days**, with additional features comparing the **most recent 30 days** to the **previous 30-day period**. These historical time windows provide the information used to assess and predict future content performance.


In [4]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the unit of analysis
df=pd.read_csv('/content/content_refresh_anonymized.csv')
print(df.columns)
print(f"Total rows: {len(df):,}")
print(f"Unique content pages: {df['content_id'].nunique():,}")

# Show the main time-window columns
time_window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d"
]

df[["content_id"] + time_window_cols].head()

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')
Total rows: 30,000
Unique content pages: 30,000


,content_id,impressions_90d,clicks_90d,sessions_90d,impressions_last_30d,impressions_prev_30d
0,content_304f48230142,3803,29,17,578,987
1,content_a1fb4e703a9e,15320,7,9,2501,5915
2,content_9aa793d4d895,12581,11,11,2382,6089
3,content_331d6c4de07b,11751,58,78,3626,4206
4,content_d99b7a2d90ca,19140,24,145,4211,6452


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The model will use historical information that is available before making a prediction. These features include search-performance metrics (`impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `ctr`, `avg_position`), engagement metrics (`engagement_rate`, `scroll_rate`, `ai_traffic_pct`), content characteristics (`word_count`, `char_count`, `content_age_days`, `days_since_last_update`), and contextual attributes such as `search_volume`, `competition`, `content_type`, and `main_intent`.

### Label

The prediction target is **`trend_direction`**, specifically whether a page shows a **downward trend** in search performance.

### Context

Columns such as `content_id` and `client_id` identify records and clients. They are useful for tracking, reporting, and presenting results but are not meaningful predictive features.

### Excluded

`trend_pct` is excluded because it directly measures the outcome associated with the prediction target and could leak information about future performance. Identifier fields (`content_id` and `client_id`) are also excluded from training because they do not represent meaningful page characteristics. Any information that would not be available at prediction time would also be excluded to prevent target leakage.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Features (examples):")
print([
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
])

print("\nLabel:")
print("trend_direction")

print("\nContext:")
print(["content_id", "client_id"])

print("\nExcluded:")
print(["trend_pct"])

Features (examples):
['search_volume', 'competition', 'word_count', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']

Label:
trend_direction

Context:
['content_id', 'client_id']

Excluded:
['trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The data contract was verified using simple checks on the dataset. First, the number of rows matches the number of unique `content_id` values, confirming that each row represents one content page. Second, the distribution of `trend_direction` confirms that the prediction label exists for all pages. Third, missing-value checks identify any incomplete feature columns that may require cleaning before modeling. Finally, the dataset contains both 90-day aggregate metrics and 30-day comparison windows, confirming that the historical time periods described in the data contract are available for analysis.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# -----------------------------
# 1. Verify the grain
# -----------------------------
print("Total rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())

# -----------------------------
# 2. Verify the label
# -----------------------------
print("\nTrend distribution:")
print(df["trend_direction"].value_counts())

# -----------------------------
# 3. Check missing values
# -----------------------------
cols_to_check = [
    "search_volume",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction"
]

print("\nMissing values:")
print(df[cols_to_check].isnull().sum())

# -----------------------------
# 4. Verify the time windows
# -----------------------------
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d"
]

print("\nTime-window columns:")
print(df[window_cols].head())

Total rows: 30000
Unique content_id: 30000

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing values:
search_volume             2468
word_count                7699
char_count                7699
impressions_90d              0
clicks_90d                   0
ctr                          0
avg_position                 0
content_age_days             0
days_since_last_update       0
trend_direction              0
dtype: int64

Time-window columns:
   impressions_90d  clicks_90d  sessions_90d  impressions_last_30d  \
0             3803          29            17                   578   
1            15320           7             9                  2501   
2            12581          11            11                  2382   
3            11751          58            78                  3626   
4            19140          24           145                  4211   

   impressions_prev_30d  clicks

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations. It contains historical search-performance summaries rather than the complete history of each webpage, so long-term trends outside the available time windows cannot be analyzed. Some metrics are aggregated over overlapping 30-day and 90-day windows, which may introduce correlations between features. The dataset is anonymized, so page content, URLs, keywords, and client-specific context are unavailable, limiting qualitative analysis. Additionally, the data shows observed relationships but cannot establish cause-and-effect or explain why a page's performance changed. Therefore, the model should be used as a **decision-support tool** rather than as proof of why Google rankings changed.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.